# Lesson 4 Data Preview: Hotel Statistics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Digital-team-repo/Data-Analysis-Using-Python/blob/main/04-excel-analysis/preview-hotel-statistics.ipynb) 
[![Open In Callysto](https://img.shields.io/badge/Open%20In-Callysto-blue)](https://hub.callysto.ca/jupyter/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2FDigital-team-repo%2FData-Analysis-Using-Python&branch=main&subPath=04-excel-analysis/preview-hotel-statistics.ipynb)

[Ontario Data Portal Page](https://data.ontario.ca/dataset/hotel-statistics)

This quick notebook helps you preview one Ontario dataset and test a non-scatter visualization option for classroom use.

## Step 1: Import libraries

In [ ]:
import io
import requests
import pandas as pd
import plotly.express as px

from urllib.parse import quote


## Step 2: Pull dataset metadata and list downloadable resources

In [ ]:
dataset_slug = "hotel-statistics"
dataset_page = "https://data.ontario.ca/dataset/hotel-statistics"

# CKAN package_show endpoint used by Ontario Data Portal
api_url = f"https://data.ontario.ca/api/3/action/package_show?id={quote(dataset_slug)}"

resp = requests.get(api_url, timeout=30)
resp.raise_for_status()
package = resp.json()["result"]

print("Dataset:", package.get("title", dataset_slug))
print("Last Updated:", package.get("metadata_modified", "unknown"))
print("Dataset Page:", dataset_page)

resources = package.get("resources", [])
print(f"Resources found: {len(resources)}")

pd.DataFrame([
    {
        "name": r.get("name"),
        "format": (r.get("format") or "").upper(),
        "url": r.get("url"),
        "last_modified": r.get("last_modified")
    }
    for r in resources
]).head(20)


## Step 3: Load one preview resource (CSV/XLSX)

In [ ]:
resource_df = pd.DataFrame(resources)
resource_df["format_upper"] = resource_df["format"].fillna("").str.upper()

# Prefer CSV first, then Excel formats
preferred = resource_df[resource_df["format_upper"].isin(["CSV", "XLSX", "XLS"])]
if preferred.empty:
    preferred = resource_df

resource = preferred.iloc[0]
resource_name = resource.get("name")
resource_format = str(resource.get("format", "")).upper()
resource_url = resource.get("url")

print("Selected resource:", resource_name)
print("Format:", resource_format)
print("URL:", resource_url)

if resource_format == "CSV":
    df = pd.read_csv(resource_url)
elif resource_format in ["XLSX", "XLS"]:
    df = pd.read_excel(resource_url)
else:
    # Try CSV first as fallback, then Excel
    try:
        df = pd.read_csv(resource_url)
    except Exception:
        df = pd.read_excel(resource_url)

print("Shape:", df.shape)
df.head()


## Step 4: Build a quick non-scatter chart

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
text_cols = df.select_dtypes(exclude="number").columns.tolist()

print("Numeric columns:", numeric_cols[:10])
print("Text/date-like columns:", text_cols[:10])

if not numeric_cols:
    raise ValueError("No numeric columns found for plotting.")

chart_hint = "bar"

# Choose reasonable defaults
y_col = numeric_cols[0]
x_col = text_cols[0] if text_cols else None

if chart_hint == "line":
    if x_col is None and len(numeric_cols) > 1:
        x_col = numeric_cols[1]
    fig = px.line(df, x=x_col, y=y_col, title=f"Quick Preview (Line): {y_col}")
elif chart_hint == "bar":
    if x_col is None and len(numeric_cols) > 1:
        x_col = numeric_cols[1]
    plot_df = df
    if x_col is not None and len(df) > 50:
        plot_df = df.head(50)
    fig = px.bar(plot_df, x=x_col, y=y_col, title=f"Quick Preview (Bar): {y_col} by {x_col}")
elif chart_hint == "box":
    if x_col is not None:
        fig = px.box(df, x=x_col, y=y_col, title=f"Quick Preview (Box): {y_col} by {x_col}")
    else:
        fig = px.box(df, y=y_col, title=f"Quick Preview (Box): {y_col}")
else:  # histogram
    fig = px.histogram(df, x=y_col, nbins=30, title=f"Quick Preview (Histogram): {y_col}")

fig.update_layout(template="plotly_white")
fig.show()
